# NanoStat-CTV - SFT + GRPO on a T4 (Colab / Kaggle)
Thin notebook: calls package code only. **NOT RUN yet** - record measured VRAM, tokens/s and losses in STATUS.md after running.
Arms with stock TRL GRPO: R-F, R-A, R-AD, R-AV. R-ADT / R-CTV need the twin-joint custom trainer (not yet implemented for the LLM).

In [ ]:
!git clone https://github.com/Anandharajan/nanostat-ctv && cd nanostat-ctv && pip install -q -e '.[gpu]'
%cd nanostat-ctv
!nvidia-smi

In [ ]:
!nanostat factory --profile full --out data
!nanostat train --stage sft-data --data data --out data/sft.jsonl

In [ ]:
!python -m training.train_gpu sft --data data --sft data/sft.jsonl --out runs/s0 --seed 42

In [ ]:
import torch

print("peak reserved GiB", torch.cuda.max_memory_reserved() / 2**30)

In [ ]:
!python -m training.train_gpu grpo --data data --sft data/sft.jsonl --init runs/s0 --arm R-F --out runs/rf --seed 42
!python -m training.train_gpu grpo --data data --sft data/sft.jsonl --init runs/s0 --arm R-AV --out runs/rav --seed 42

Evaluate each adapter by serving it locally (merge LoRA, convert to GGUF, load in Magnitude) and running:
`nanostat eval --arms B1 --llm-model <id> --data data --split test` and `nanostat eval --arms B1 --suite pbench --llm-model <id> --trials 3`.